In [ ]:
#Task 1
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1.1 Ingest & Clean (Using Synthetic Data for testing)
# ==========================================
print("1. Generating Synthetic Market Data...")
n_assets = 50
n_days = 2520  # Roughly 10 years of daily data
lookback = 30

# Simulate daily log returns for 50 assets
np.random.seed(42)
returns_data = np.random.normal(loc=0.0002, scale=0.015, size=(n_days, n_assets))
asset_names = [f'Asset_{i}' for i in range(n_assets)]

# Create a Pandas data frame
df_returns = pd.DataFrame(returns_data, columns=asset_names)

# ==========================================
# 1.2 Feature selection
# ==========================================
print("2. Features selection")
# Calculate 20-day rolling realized volatility (standard deviation)
df_volatility = df_returns.rolling(window=20).std()

# Combine returns and volatility into one massive feature matrix
# Drop the first 19 rows which contain NaNs due to the rolling window
df_features = pd.concat([df_returns, df_volatility.add_suffix('_vol')], axis=1).dropna()
df_returns = df_returns.loc[df_features.index] # Align targets with dropped NaNs

print(f"Feature Matrix Shape: {df_features.shape}") # Should be (2501, 100)

# ==========================================
# 1.3 Chronological Split & Scaling (Crucial Step)
# ==========================================
print("3. Chronological Splitting and Scaling...")
# Split data: 80% Train, 20% Test (NEVER shuffle time-series data)
split_idx = int(len(df_features) * 0.8)

train_features = df_features.iloc[:split_idx]
test_features = df_features.iloc[split_idx:]

# Target variable (Y) is JUST the returns. We do NOT scale Y because
# the covariance matrix in the ADMM solver needs true market scale.
train_targets = df_returns.iloc[:split_idx]
test_targets = df_returns.iloc[split_idx:]

# Fit scaler ONLY on training data to prevent look-ahead bias
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)
test_features_scaled = scaler.transform(test_features) # Transform test with train's parameters

# ==========================================
# 1.4 Tensor Structuring (Sliding Window)
# ==========================================
print("4. Structuring 3D Tensors for Keras...")

def create_tensor_windows(features_array, targets_array, lookback):
    """
    Slides a window of size 'lookback' over the data.
    X shape: (samples, lookback, num_features)
    Y shape: (samples, n_assets) -> The return at t+1
    """
    X, Y = [], []
    # Loop stops at len - lookback so we always have a t+1 target
    for i in range(len(features_array) - lookback):
        # The sequence of 'lookback' days (e.g., day 0 to 29)
        X.append(features_array[i : i + lookback])
        # The target is the return on the very next day (e.g., day 30)
        Y.append(targets_array.iloc[i + lookback].values)

    return np.array(X), np.array(Y)

# Generate Train Tensors
X_train, Y_train = create_tensor_windows(train_features_scaled, train_targets, lookback)

# Generate Test Tensors
X_test, Y_test = create_tensor_windows(test_features_scaled, test_targets, lookback)

# ==========================================
# 1.5 Verification
# ==========================================
print("\n--- Final Tensor Architecture ---")
print(f"X_train shape: {X_train.shape} -> (batch_size, time_steps, features)")
print(f"Y_train shape: {Y_train.shape}   -> (batch_size, n_assets)")
print(f"X_test shape : {X_test.shape}")
print(f"Y_test shape : {Y_test.shape}")

1. Generating Synthetic Market Data...
2. Features selection
Feature Matrix Shape: (2501, 100)
3. Chronological Splitting and Scaling...
4. Structuring 3D Tensors for Keras...

--- Final Tensor Architecture ---
X_train shape: (1970, 30, 100) -> (batch_size, time_steps, features)
Y_train shape: (1970, 50)   -> (batch_size, n_assets)
X_test shape : (471, 30, 100)
Y_test shape : (471, 50)


In [ ]:
def build_parameter_engine(n_assets, lookback, num_features, k_factors=5):
    # ==========================================
    # 1. Shared Sequence Learner
    # ==========================================
    # Input shape corresponds to (30, 100) from Task 1
    inputs = Input(shape=(lookback, num_features), name='market_state_input')

    # Process the sequence to extract a unified latent representation
    x = LSTM(64, return_sequences=False)(inputs)
    shared_dense = Dense(64, activation='swish')(x)

    # ==========================================
    # 2. Head A: Expected Returns (Mu)
    # ==========================================
    # Linear activation because expected returns can be positive or negative
    mu_out = Dense(n_assets, activation='linear', name='mu_head')(shared_dense)

    # ==========================================
    # 3. Head B: Conditional Covariance (Sigma)
    # ==========================================
    # 3.1 Map to the factor loading matrix (A)
    # We flatten it first, then reshape it to (n, k)
    A_flat = Dense(n_assets * k_factors, activation='linear')(shared_dense)
    A_matrix = Reshape((n_assets, k_factors))(A_flat)

    # 3.2 Map to idiosyncratic variance (D)
    # CRITICAL: We use 'softplus' activation to force D to be strictly positive
    D_flat = Dense(n_assets, activation='softplus')(shared_dense)

    # 3.3 Pass through our custom SPD assembly layer
    sigma_out = SPDCovarianceLayer(n_assets=n_assets, name='sigma_head')([A_matrix, D_flat])

    # ==========================================
    # 4. Compile Architecture
    # ==========================================
    model = Model(inputs=inputs, outputs=[mu_out, sigma_out], name="LSTM_Parameter_Engine")

    return model

# Instantiate the model based on Task 1 parameters
n_assets = 50
lookback = 30
num_features = 100 # 50 returns + 50 volatilities

model = build_parameter_engine(n_assets, lookback, num_features, k_factors=5)
model.summary()

Model: "LSTM_Parameter_Engine"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ market_state_input  │ (None, 30, 100)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 64)        │     42,240 │ market_state_inp… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 250)       │     16,250 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 50, 5)     │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 50)        │      3,250 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mu_head (Dense)     │ (None, 50)        │      3,250 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sigma_head          │ (None, 50, 50)    │          0 │ reshape[0][0],    │
│ (SPDCovarianceLaye… │                   │            │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 69,150 (270.12 KB)

 Trainable params: 69,150 (270.12 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import tensorflow as tf

# ==========================================
# 3.1 The Custom NLL Loss Function
# ==========================================
def nll_loss(y_true, mu_pred, sigma_pred):
    """
    y_true: Realized returns at t+1 (batch_size, n_assets)
    mu_pred: Predicted expected returns (batch_size, n_assets)
    sigma_pred: Predicted covariance matrix (batch_size, n_assets, n_assets)
    """
    # 1. Calculate residuals: (r - mu)
    # Expand dims to make it a column vector for matrix multiplication: (batch, n, 1)
    residuals = tf.expand_dims(y_true - mu_pred, axis=-1)

    # 2. Log Determinant of Sigma: ln|Sigma|
    # We use slogdet to avoid underflow/overflow. It returns (sign, log_det)
    _, log_det = tf.linalg.slogdet(sigma_pred)

    # 3. The Quadratic Form: (r - mu)^T * Sigma^-1 * (r - mu)
    # Instead of inverting Sigma, we solve Sigma * x = residuals for x.
    # This means x = Sigma^-1 * residuals.
    solve_x = tf.linalg.solve(sigma_pred, residuals)

    # Now multiply residuals^T * x
    # transpose_a=True transposes the residuals tensor -> (batch, 1, n)
    quad_form = tf.matmul(residuals, solve_x, transpose_a=True)

    # Remove extra dimensions -> shape becomes (batch_size,)
    quad_form = tf.squeeze(quad_form, axis=[-1, -2])

    # 4. Total Loss
    loss = 0.5 * log_det + 0.5 * quad_form

    # Return the mean loss across the batch
    return tf.reduce_mean(loss)

# ==========================================
# 3.2 The Custom Keras Model Subclass (The Pro Way)
# ==========================================
# By subclassing tf.keras.Model, we dictate exactly how the gradients are applied.
class ParameterEngine(tf.keras.Model):
    def __init__(self, base_model, **kwargs):
        super(ParameterEngine, self).__init__(**kwargs)
        self.base_model = base_model

    def call(self, inputs, training=False):
        return self.base_model(inputs, training=training)

    def train_step(self, data):
        x, y = data # x is X_train (market state), y is Y_train (realized returns)

        with tf.GradientTape() as tape:
            # Forward pass: extract mu and sigma
            mu_pred, sigma_pred = self.base_model(x, training=True)

            # Compute our custom mathematical loss
            loss = nll_loss(y, mu_pred, sigma_pred)

        # Compute gradients and apply them via the optimizer
        trainable_vars = self.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)

        # Gradient clipping is highly recommended for covariance matrix training
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)

        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        return {"nll_loss": loss}

    def test_step(self, data):
        # Used during validation
        x, y = data
        mu_pred, sigma_pred = self.base_model(x, training=False)
        loss = nll_loss(y, mu_pred, sigma_pred)
        return {"val_nll_loss": loss}

In [ ]:
# Assuming 'model' is the Keras Model graph we built in Task 2
# Wrap it in our custom training engine
custom_engine = ParameterEngine(base_model=model)

# Compile using a small learning rate. NLL is highly sensitive initially.
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
custom_engine.compile(optimizer=optimizer)

print("Starting Training Sequence...")

# Train the model (Using the X_train, Y_train from Task 1)
history = custom_engine.fit(
    x=X_train,
    y=Y_train,
    validation_data=(X_test, Y_test),
    batch_size=64,
    epochs=50,
    callbacks=[
        # Stop early if validation loss starts climbing
        tf.keras.callbacks.EarlyStopping(monitor='val_nll_loss', patience=10, restore_best_weights=True)
    ]
)

print("Training Complete. Generating final parameter matrices for ADMM solver.")

# Extract the final predicted parameters for the test set
# These are the matrices that will feed into the optimization engine!
predicted_mu, predicted_sigma = custom_engine.predict(X_test)

print(f"Predicted Mu Shape: {predicted_mu.shape}")       # Should be (test_samples, 50)
print(f"Predicted Sigma Shape: {predicted_sigma.shape}") # Should be (test_samples, 50, 50)

Starting Training Sequence...
Epoch 1/50
30/31 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - nll_loss: -7.9903

ValueError: EarlyStopping callback received monitor=val_nll_loss, but Keras isn't able to automatically determine whether that metric should be maximized or minimized. Pass `mode='max'` in order to monitor based on the highest metric value, or pass `mode='min'` in order to use the lowest value.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Reshape, Layer
from tensorflow.keras.models import Model

print("TF Version:", tf.__version__)

# ==========================================
# TASK 1: DATA ENGINEERING & TENSOR FORMATTING
# ==========================================
print("\n--- Executing Task 1: Data Engineering ---")
n_assets = 50
n_days = 2520  # Roughly 10 years
lookback = 30

# 1. Generate Synthetic Market Data
np.random.seed(42)
tf.random.set_seed(42)
returns_data = np.random.normal(loc=0.0002, scale=0.015, size=(n_days, n_assets))
df_returns = pd.DataFrame(returns_data, columns=[f'Asset_{i}' for i in range(n_assets)])

# 2. Rolling Volatility Features
df_volatility = df_returns.rolling(window=20).std()
df_features = pd.concat([df_returns, df_volatility.add_suffix('_vol')], axis=1).dropna()
df_returns = df_returns.loc[df_features.index]

# 3. Chronological Split & Scaling
split_idx = int(len(df_features) * 0.8)
train_features = df_features.iloc[:split_idx]
test_features = df_features.iloc[split_idx:]
train_targets = df_returns.iloc[:split_idx]
test_targets = df_returns.iloc[split_idx:]

scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)
test_features_scaled = scaler.transform(test_features)

# 4. Create 3D Tensors
def create_tensor_windows(features_array, targets_array, lookback):
    X, Y = [], []
    for i in range(len(features_array) - lookback):
        X.append(features_array[i : i + lookback])
        Y.append(targets_array.iloc[i + lookback].values)
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X_train, Y_train = create_tensor_windows(train_features_scaled, train_targets, lookback)
X_test, Y_test = create_tensor_windows(test_features_scaled, test_targets, lookback)

print(f"X_train shape generated: {X_train.shape}")
print(f"Y_train shape generated: {Y_train.shape}")


# ==========================================
# TASK 2: ARCHITECTURE & CUSTOM SPD LAYER
# ==========================================
print("\n--- Executing Task 2: Building Architecture ---")

class SPDCovarianceLayer(Layer):
    def __init__(self, n_assets, epsilon=1e-5, **kwargs):
        super(SPDCovarianceLayer, self).__init__(**kwargs)
        self.n_assets = n_assets
        self.epsilon = epsilon

    def call(self, inputs):
        A_matrix, D_vector = inputs
        A_AT = tf.matmul(A_matrix, A_matrix, transpose_b=True)
        D_diag = tf.linalg.diag(D_vector)
        stability_eye = self.epsilon * tf.eye(self.n_assets, dtype=tf.float32)
        return A_AT + D_diag + stability_eye

def build_parameter_engine(n_assets, lookback, num_features, k_factors=5):
    inputs = Input(shape=(lookback, num_features), name='market_state_input')
    x = LSTM(64, return_sequences=False)(inputs)
    shared_dense = Dense(64, activation='swish')(x)

    # Head A: Expected Returns
    mu_out = Dense(n_assets, activation='linear', name='mu_head')(shared_dense)

    # Head B: SPD Covariance Matrix
    A_flat = Dense(n_assets * k_factors, activation='linear')(shared_dense)
    A_matrix = Reshape((n_assets, k_factors))(A_flat)
    D_flat = Dense(n_assets, activation='softplus')(shared_dense)
    sigma_out = SPDCovarianceLayer(n_assets=n_assets, name='sigma_head')([A_matrix, D_flat])

    return Model(inputs=inputs, outputs=[mu_out, sigma_out], name="LSTM_Parameter_Engine")

num_features = X_train.shape[2]
base_model = build_parameter_engine(n_assets, lookback, num_features, k_factors=5)


# ==========================================
# TASK 3: CUSTOM NLL LOSS & TRAINING LOOP
# ==========================================
print("\n--- Executing Task 3: Training the Model ---")

def nll_loss(y_true, mu_pred, sigma_pred):
    residuals = tf.expand_dims(y_true - mu_pred, axis=-1)
    _, log_det = tf.linalg.slogdet(sigma_pred)
    solve_x = tf.linalg.solve(sigma_pred, residuals)
    quad_form = tf.matmul(residuals, solve_x, transpose_a=True)
    quad_form = tf.squeeze(quad_form, axis=[-1, -2])
    loss = 0.5 * log_det + 0.5 * quad_form
    return tf.reduce_mean(loss)

class ParameterEngine(tf.keras.Model):
    def __init__(self, base_model, **kwargs):
        super(ParameterEngine, self).__init__(**kwargs)
        self.base_model = base_model

    def call(self, inputs, training=False):
        return self.base_model(inputs, training=training)

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            mu_pred, sigma_pred = self.base_model(x, training=True)
            loss = nll_loss(y, mu_pred, sigma_pred)

        trainable_vars = self.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        return {"nll_loss": loss}

    def test_step(self, data):
        x, y = data
        mu_pred, sigma_pred = self.base_model(x, training=False)
        loss = nll_loss(y, mu_pred, sigma_pred)
        return {"val_nll_loss": loss}

# 1. Wrap the base model
custom_engine = ParameterEngine(base_model=base_model)

# 2. Compile
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
custom_engine.compile(optimizer=optimizer)

# 3. Train (Using 10 epochs for speed, increase this later)
print("Starting Training Sequence... (This will take a moment)")
history = custom_engine.fit(
    x=X_train,
    y=Y_train,
    validation_data=(X_test, Y_test),
    batch_size=64,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_nll_loss', patience=3, restore_best_weights=True, mode='min')
    ]
)

# 4. Extract Final Predictions
print("\n--- Generating Final Parameters for ADMM ---")
predicted_mu, predicted_sigma = custom_engine.predict(X_test)

print(f"Final Predicted Mu Shape: {predicted_mu.shape} -> Ready for ADMM")
print(f"Final Predicted Sigma Shape: {predicted_sigma.shape} -> Ready for ADMM")

TF Version: 2.20.0

--- Executing Task 1: Data Engineering ---
X_train shape generated: (1970, 30, 100)
Y_train shape generated: (1970, 50)

--- Executing Task 2: Building Architecture ---

--- Executing Task 3: Training the Model ---
Starting Training Sequence... (This will take a moment)
Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - nll_loss: -7.8048 - val_val_nll_loss: -7.7096
Epoch 2/10
 3/31 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - nll_loss: -7.7640

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_nll_loss` which is not available. Available metrics are: nll_loss,val_val_nll_loss
  current = self.get_monitor_value(logs)


31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step - nll_loss: -8.2661 - val_val_nll_loss: -8.0870
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - nll_loss: -8.7163 - val_val_nll_loss: -8.4731
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - nll_loss: -9.2278 - val_val_nll_loss: -8.9240
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - nll_loss: -9.9122 - val_val_nll_loss: -9.5376
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - nll_loss: -10.9784 - val_val_nll_loss: -10.5264
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - nll_loss: -12.8747 - val_val_nll_loss: -12.4546
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - nll_loss: -16.6927 - val_val_nll_loss: -17.1175
Epoch 9/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - nll_loss: -25.4443 - val_val_nll_loss: -28.2804
Epoch 10/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step - nll_loss: -45.2576 - val_val_nll_loss: -48.0977

--- Generating Final Parameters for ADMM ---
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step
Final Predicted Mu Shap

In [ ]:
import numpy as np

# ==========================================
# 1. SIMPLEX PROJECTION ALGORITHM
# ==========================================
def project_simplex(v):
    """
    Projects a vector v onto the probability simplex: sum(x) = 1, x >= 0.
    This is an exact O(N log N) continuous sorting algorithm.
    """
    n = v.shape[0]
    # Sort v in descending order
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    # Find the optimal threshold
    rho = np.nonzero(u * np.arange(1, n+1) > (cssv - 1))[0][-1]
    theta = (cssv[rho] - 1) / (rho + 1.0)
    # Apply soft clipping
    return np.maximum(v - theta, 0)

# ==========================================
# 2. ADMM PROXIMAL SOLVER ENGINE
# ==========================================
def admm_multi_period_optimizer(mu_seq, sigma_seq, x0, gamma=2.0, lamda=0.005, rho=1.0, max_iter=200, tol=1e-4):
    """
    mu_seq: Expected returns (T, N)
    sigma_seq: Covariance matrices (T, N, N)
    x0: Initial portfolio weights (N,)
    gamma: Risk aversion scalar
    lamda: L1 transaction cost penalty (larger = sparser trading)
    rho: ADMM learning rate / quadratic penalty
    """
    T, n = mu_seq.shape

    # Initialize Primal and Dual Variables
    x = np.ones((T, n)) / n  # Initialize with equal weights
    u = np.zeros((T, n))     # Trading volumes
    y = np.zeros((T, n))     # Dual multipliers

    # Precompute Q inverses (since Sigma is constant per time step)
    Q_inv = np.zeros((T, n, n))
    I = np.eye(n)
    for t in range(T):
        if t < T - 1:
            Q = gamma * sigma_seq[t] + 2 * rho * I
        else:
            Q = gamma * sigma_seq[t] + rho * I
        # Stable inversion
        Q_inv[t] = np.linalg.inv(Q)

    print(f"Beginning ADMM Iterations for {T} periods...")

    for k in range(max_iter):
        x_old = np.copy(x)
        u_old = np.copy(u)

        # -----------------------------------
        # Block 1: x-update (Gauss-Seidel sequential sweep)
        # -----------------------------------
        for t in range(T):
            x_prev = x[t-1] if t > 0 else x0

            if t < T - 1:
                x_next = x[t+1]
                # c_t contains pull from yesterday AND tomorrow
                c = mu_seq[t] + \
                    rho * (x_prev + u[t] - y[t]/rho) + \
                    rho * (x_next - u[t+1] + y[t+1]/rho)
            else:
                # Last period only pulls from yesterday
                c = mu_seq[t] + rho * (x_prev + u[t] - y[t]/rho)

            # Unconstrained solution
            x_unc = Q_inv[t] @ c

            # Project onto strict portfolio simplex
            x[t] = project_simplex(x_unc)

        # -----------------------------------
        # Block 2: u-update (Soft Thresholding Proximal Mapping)
        # -----------------------------------
        for t in range(T):
            x_prev = x[t-1] if t > 0 else x0
            # Target velocity
            v_t = x[t] - x_prev + y[t]/rho

            # Analytical Soft-Thresholding for L1 norm
            kappa = lamda / rho
            u[t] = np.sign(v_t) * np.maximum(np.abs(v_t) - kappa, 0)

        # -----------------------------------
        # Block 3: y-update (Dual Ascent)
        # -----------------------------------
        for t in range(T):
            x_prev = x[t-1] if t > 0 else x0
            y[t] = y[t] + rho * (x[t] - x_prev - u[t])

        # -----------------------------------
        # Convergence Check (Primal and Dual Residuals)
        # -----------------------------------
        primal_res = 0.0
        dual_res = 0.0
        for t in range(T):
            x_prev = x[t-1] if t > 0 else x0
            # Primal error: Does x_t - x_{t-1} actually equal u_t?
            primal_res += np.linalg.norm(x[t] - x_prev - u[t])**2
            # Dual error: Has u stabilized?
            dual_res += np.linalg.norm(rho * (u[t] - u_old[t]))**2

        primal_res = np.sqrt(primal_res)
        dual_res = np.sqrt(dual_res)

        if (k+1) % 50 == 0 or k == 0:
            print(f"Iter {k+1:3d} | Primal Res: {primal_res:.6f} | Dual Res: {dual_res:.6f}")

        if primal_res < tol and dual_res < tol:
            print(f"ADMM Converged EXACTLY at iteration {k+1}")
            break

    return x, u

In [ ]:
import numpy as np

# ==========================================
# 1. SIMPLEX PROJECTION ALGORITHM
# ==========================================
def project_simplex(v):
    """
    Projects a vector v onto the probability simplex: sum(x) = 1, x >= 0.
    This is an exact O(N log N) continuous sorting algorithm.
    """
    n = v.shape[0]
    # Sort v in descending order
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    # Find the optimal threshold
    rho = np.nonzero(u * np.arange(1, n+1) > (cssv - 1))[0][-1]
    theta = (cssv[rho] - 1) / (rho + 1.0)
    # Apply soft clipping
    return np.maximum(v - theta, 0)

# ==========================================
# 2. ADMM PROXIMAL SOLVER ENGINE
# ==========================================
def admm_multi_period_optimizer(mu_seq, sigma_seq, x0, gamma=2.0, lamda=0.005, rho=1.0, max_iter=200, tol=1e-4):
    """
    mu_seq: Expected returns (T, N)
    sigma_seq: Covariance matrices (T, N, N)
    x0: Initial portfolio weights (N,)
    gamma: Risk aversion scalar
    lamda: L1 transaction cost penalty (larger = sparser trading)
    rho: ADMM learning rate / quadratic penalty
    """
    T, n = mu_seq.shape

    # Initialize Primal and Dual Variables
    x = np.ones((T, n)) / n  # Initialize with equal weights
    u = np.zeros((T, n))     # Trading volumes
    y = np.zeros((T, n))     # Dual multipliers

    # Precompute Q inverses (since Sigma is constant per time step)
    Q_inv = np.zeros((T, n, n))
    I = np.eye(n)
    for t in range(T):
        if t < T - 1:
            Q = gamma * sigma_seq[t] + 2 * rho * I
        else:
            Q = gamma * sigma_seq[t] + rho * I
        # Stable inversion
        Q_inv[t] = np.linalg.inv(Q)

    print(f"Beginning ADMM Iterations for {T} periods...")

    for k in range(max_iter):
        x_old = np.copy(x)
        u_old = np.copy(u)

        # -----------------------------------
        # Block 1: x-update (Gauss-Seidel sequential sweep)
        # -----------------------------------
        for t in range(T):
            x_prev = x[t-1] if t > 0 else x0

            if t < T - 1:
                x_next = x[t+1]
                # c_t contains pull from yesterday AND tomorrow
                c = mu_seq[t] + \
                    rho * (x_prev + u[t] - y[t]/rho) + \
                    rho * (x_next - u[t+1] + y[t+1]/rho)
            else:
                # Last period only pulls from yesterday
                c = mu_seq[t] + rho * (x_prev + u[t] - y[t]/rho)

            # Unconstrained solution
            x_unc = Q_inv[t] @ c

            # Project onto strict portfolio simplex
            x[t] = project_simplex(x_unc)

        # -----------------------------------
        # Block 2: u-update (Soft Thresholding Proximal Mapping)
        # -----------------------------------
        for t in range(T):
            x_prev = x[t-1] if t > 0 else x0
            # Target velocity
            v_t = x[t] - x_prev + y[t]/rho

            # Analytical Soft-Thresholding for L1 norm
            kappa = lamda / rho
            u[t] = np.sign(v_t) * np.maximum(np.abs(v_t) - kappa, 0)

        # -----------------------------------
        # Block 3: y-update (Dual Ascent)
        # -----------------------------------
        for t in range(T):
            x_prev = x[t-1] if t > 0 else x0
            y[t] = y[t] + rho * (x[t] - x_prev - u[t])

        # -----------------------------------
        # Convergence Check (Primal and Dual Residuals)
        # -----------------------------------
        primal_res = 0.0
        dual_res = 0.0
        for t in range(T):
            x_prev = x[t-1] if t > 0 else x0
            # Primal error: Does x_t - x_{t-1} actually equal u_t?
            primal_res += np.linalg.norm(x[t] - x_prev - u[t])**2
            # Dual error: Has u stabilized?
            dual_res += np.linalg.norm(rho * (u[t] - u_old[t]))**2

        primal_res = np.sqrt(primal_res)
        dual_res = np.sqrt(dual_res)

        if (k+1) % 50 == 0 or k == 0:
            print(f"Iter {k+1:3d} | Primal Res: {primal_res:.6f} | Dual Res: {dual_res:.6f}")

        if primal_res < tol and dual_res < tol:
            print(f"ADMM Converged EXACTLY at iteration {k+1}")
            break

    return x, u

In [ ]:
# Assuming predicted_mu and predicted_sigma are loaded in memory from Task 3
# Their shapes should be (T_test, N_assets) and (T_test, N_assets, N_assets)

# Define an initial starting portfolio (e.g., equally weighted)
N_assets = predicted_mu.shape[1]
x_initial = np.ones(N_assets) / N_assets

# Set your Hyperparameters
# Gamma determines risk aversion (higher = lower risk).
# Lamda determines turnover aversion (higher = less trading).
GAMMA = 5.0
LAMDA = 0.005 # Play with this! Make it 0.05 to see trading stop completely.

# Run the Optimizer
optimal_weights, optimal_trades = admm_multi_period_optimizer(
    mu_seq=predicted_mu,
    sigma_seq=predicted_sigma,
    x0=x_initial,
    gamma=GAMMA,
    lamda=LAMDA,
    rho=1.0 # ADMM learning rate, usually 1.0 is fine
)

print("\n--- FINAL RESULTS ---")
print("Total number of time steps optimized:", optimal_weights.shape[0])

# Let's look at the mathematical sparsity induced by the L1 norm
# Any trade smaller than 1e-6 is considered perfectly zero.
zero_trades = np.sum(np.abs(optimal_trades) <= 1e-6)
total_trades = optimal_trades.size
sparsity_pct = (zero_trades / total_trades) * 100

print(f"\nExecution Sparsity Check:")
print(f"Total possible asset trades: {total_trades}")
print(f"Trades suppressed to EXACTLY zero: {zero_trades}")
print(f"Sparsity Percentage: {sparsity_pct:.2f}%")

NameError: name 'predicted_mu' is not defined

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Reshape, Layer
from tensorflow.keras.models import Model

# ==========================================
# TASK 1: DATA ENGINEERING & TENSOR FORMATTING
# ==========================================
print("--- TASK 1: Generating Market Data ---")
n_assets = 50
n_days = 2520
lookback = 30

np.random.seed(42)
tf.random.set_seed(42)
returns_data = np.random.normal(loc=0.0002, scale=0.015, size=(n_days, n_assets))
df_returns = pd.DataFrame(returns_data, columns=[f'Asset_{i}' for i in range(n_assets)])

df_volatility = df_returns.rolling(window=20).std()
df_features = pd.concat([df_returns, df_volatility.add_suffix('_vol')], axis=1).dropna()
df_returns = df_returns.loc[df_features.index]

split_idx = int(len(df_features) * 0.8)
train_features = df_features.iloc[:split_idx]
test_features = df_features.iloc[split_idx:]
train_targets = df_returns.iloc[:split_idx]
test_targets = df_returns.iloc[split_idx:]

scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)
test_features_scaled = scaler.transform(test_features)

def create_tensor_windows(features_array, targets_array, lookback):
    X, Y = [], []
    for i in range(len(features_array) - lookback):
        X.append(features_array[i : i + lookback])
        Y.append(targets_array.iloc[i + lookback].values)
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X_train, Y_train = create_tensor_windows(train_features_scaled, train_targets, lookback)
X_test, Y_test = create_tensor_windows(test_features_scaled, test_targets, lookback)


# ==========================================
# TASK 2 & 3: LSTM ARCHITECTURE & TRAINING
# ==========================================
print("\n--- TASK 2 & 3: Building and Training LSTM ---")

class SPDCovarianceLayer(Layer):
    def __init__(self, n_assets, epsilon=1e-5, **kwargs):
        super(SPDCovarianceLayer, self).__init__(**kwargs)
        self.n_assets = n_assets
        self.epsilon = epsilon

    def call(self, inputs):
        A_matrix, D_vector = inputs
        A_AT = tf.matmul(A_matrix, A_matrix, transpose_b=True)
        D_diag = tf.linalg.diag(D_vector)
        stability_eye = self.epsilon * tf.eye(self.n_assets, dtype=tf.float32)
        return A_AT + D_diag + stability_eye

def build_parameter_engine(n_assets, lookback, num_features, k_factors=5):
    inputs = Input(shape=(lookback, num_features))
    x = LSTM(64, return_sequences=False)(inputs)
    shared_dense = Dense(64, activation='swish')(x)

    mu_out = Dense(n_assets, activation='linear', name='mu_head')(shared_dense)

    A_flat = Dense(n_assets * k_factors, activation='linear')(shared_dense)
    A_matrix = Reshape((n_assets, k_factors))(A_flat)
    D_flat = Dense(n_assets, activation='softplus')(shared_dense)
    sigma_out = SPDCovarianceLayer(n_assets=n_assets, name='sigma_head')([A_matrix, D_flat])

    return Model(inputs=inputs, outputs=[mu_out, sigma_out])

def nll_loss(y_true, mu_pred, sigma_pred):
    residuals = tf.expand_dims(y_true - mu_pred, axis=-1)
    _, log_det = tf.linalg.slogdet(sigma_pred)
    solve_x = tf.linalg.solve(sigma_pred, residuals)
    quad_form = tf.matmul(residuals, solve_x, transpose_a=True)
    quad_form = tf.squeeze(quad_form, axis=[-1, -2])
    return tf.reduce_mean(0.5 * log_det + 0.5 * quad_form)

class ParameterEngine(tf.keras.Model):
    def __init__(self, base_model, **kwargs):
        super(ParameterEngine, self).__init__(**kwargs)
        self.base_model = base_model

    def call(self, inputs, training=False):
        return self.base_model(inputs, training=training)

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            mu_pred, sigma_pred = self.base_model(x, training=True)
            loss = nll_loss(y, mu_pred, sigma_pred)
        trainable_vars = self.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        return {"nll_loss": loss}

    def test_step(self, data):
        x, y = data
        mu_pred, sigma_pred = self.base_model(x, training=False)
        return {"val_nll_loss": nll_loss(y, mu_pred, sigma_pred)}

num_features = X_train.shape[2]
base_model = build_parameter_engine(n_assets, lookback, num_features)
custom_engine = ParameterEngine(base_model=base_model)
custom_engine.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4))

print("Training for 5 epochs (accelerated for demonstration)...")
custom_engine.fit(X_train, Y_train, validation_data=(X_test, Y_test), batch_size=64, epochs=5, verbose=1)

# >>> THIS IS WHERE THE VARIABLES ARE CREATED <<<
print("\nExtracting predicted parameters for optimization...")
predicted_mu, predicted_sigma = custom_engine.predict(X_test)


# ==========================================
# TASK 4: ADMM PROXIMAL SOLVER (MATH)
# ==========================================
print("\n--- TASK 4: ADMM Portfolio Optimization ---")

def project_simplex(v):
    n = v.shape[0]
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho = np

--- TASK 1: Generating Market Data ---

--- TASK 2 & 3: Building and Training LSTM ---
Training for 5 epochs (accelerated for demonstration)...
Epoch 1/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - nll_loss: -7.7596 - val_val_nll_loss: -7.1417
Epoch 2/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - nll_loss: -8.2665 - val_val_nll_loss: -7.6972
Epoch 3/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - nll_loss: -8.7703 - val_val_nll_loss: -8.2562
Epoch 4/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - nll_loss: -9.3636 - val_val_nll_loss: -8.9184
Epoch 5/5
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - nll_loss: -10.1869 - val_val_nll_loss: -9.8456

Extracting predicted parameters for optimization...
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step

--- TASK 4: ADMM Portfolio Optimization ---
